## Setup

In [4]:
# Put all imports here #######################
import sys, json
from pathlib import Path
import numpy as np
import torch
from rdkit import Chem

##############################################

SRC = next((p / "src" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "lead_optimization.py").is_file()), None)
if SRC is None:
    raise RuntimeError(
        "cannot find lead_optimization.py -- run this notebook from inside the "
        "MolPLAtte repo, or set SRC to <repo>/molplatte/src by hand")
sys.path.insert(0, str(SRC))
print("src:", SRC)


from lead_optimization import LeadOptimizer, pocket_embedding_from_structure
from lead_report import (run_lead_optimization, PERMISSIBLE_FLAVORS,
                         TASTE_FLAVORS, ODOUR_FLAVORS)

DATA       = Path.home() / "preprocessed" / "molplatte"
CHECKPOINT = Path.home() / "checkpoints" / "molplatte" / "molplatte-final-s911012.pt"
VOCAB      = DATA / "union_vocab" / "base-full__crossdocked__tastepocket" / "rgroup_vocab.pkl.gz"

opt = LeadOptimizer.load(CHECKPOINT, VOCAB, device="cuda")
opt.build_library(batch_size=1024)

print(f"library rows      {len(opt.vocab):,}")
print(f"effective size    {opt.vocab.effective_size():.0f}")
print(f"top-1 share       {100 * opt.vocab.frequency_prior[0]:.2f}%")
print(f"novel rows        {len(opt._novel):,}  (absent from the pretraining vocabulary)")

# The permissible flavour strings -- the 24 condvec bits the corpus stores.
print(f"\nflavours ({len(PERMISSIBLE_FLAVORS)}):")
print("  taste:", ", ".join(TASTE_FLAVORS))
print("  odour:", ", ".join(ODOUR_FLAVORS))
print("  other: odorless, unknown")

[LeadOptimizer] 6 tensors missing from the checkpoint and left at init: ['nnet.pocket_conditioning.project.0.weight', 'nnet.pocket_conditioning.project.0.bias', 'nnet.pocket_conditioning.project.1.weight']


src: /home/mogan/github/MolPLAtte/molplatte/src
library rows      91,935
effective size    946
top-1 share       10.42%
novel rows        5,550  (absent from the pretraining vocabulary)

flavours (24):
  taste: sweet, bitter, sour, salty, umami
  odour: fruity, green, floral, fatty, woody, spicy, roasted, sulfurous, earthy, nutty, herbal, medicinal, citrus, dairy, alcoholic, meaty, minty
  other: odorless, unknown


## Function Definitions for Lead Optimization

In [5]:
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
from rdkit import Chem

from lead_optimization import LeadOptimizer
from lead_report import run_lead_optimization


def lead_optimization_pocketless(
    input_compound:   str | Chem.Mol,      # SMILES string or RDKit Mol
    lead_optimizer:   LeadOptimizer,
    flavor_condition: list[str],           # from PERMISSIBLE_FLAVORS
    top_k:            int = 10,
    max_decompositions: int = 4,
    gallery:          Optional[str | Path] = "lead_optimization.pdf",
):
    """Flavour-conditioned lead optimization, scored and rendered.

    Returns a LeadOptimizationReport:

        .table      deduplicated products, best score first, with
                    retrieval_score / MW / logP / QED / SAScore / NPScore,
                    plus d<prop> = product - input for each of those
        .reference  the INPUT compound's own specs, same five properties
        .compounds  the product SMILES, in table order
        .failures   suggestions that could not be assembled, WITH the reason
        .gallery    path to the PDF/PNG, or None if drawing was unavailable
        .results    the raw per-slot SlotResult list

    `retrieval_score` is the logQ-corrected value that actually ranks
    (sim/tau + log p(k)), not a similarity -- do not re-sort the table by
    anything else and expect the model's ordering back.

    Products are keyed on canonical SMILES across every decomposition and slot,
    so the same molecule proposed from three slots is ONE compound with
    n_slots=3, rather than three.

    `max_decompositions` is how many ways the input is cut into core + R-groups
    before any retrieval happens. It is the main control on how much comes back:
    the work is roughly max_decompositions x slots-per-decomposition x top_k, so
    raising it explores more of the molecule and costs proportionally. 1 uses
    only the highest-ranked decomposition.
    """
    return run_lead_optimization(
        input_compound, lead_optimizer, flavor_condition,
        top_k=top_k, max_decompositions=max_decompositions,
        pocket_condition=None, gallery=gallery,
    )


def lead_optimization_pocket(
    input_compound:   str | Chem.Mol,
    lead_optimizer:   LeadOptimizer,
    flavor_condition: list[str],
    pocket_condition: np.ndarray,          # 1280-d ESM-2 pocket embedding
    top_k:            int = 10,
    max_decompositions: int = 4,
    gallery:          Optional[str | Path] = "lead_optimization_pocket.pdf",
):
    """The same, additionally conditioned on a pocket.

    Get `pocket_condition` from a structure with

        pocket_embedding_from_structure(Path("receptor.cif"), device="cuda")

    Calculate the vina-related scores additionally, and also render the docking poses (if possible)

    """
    return run_lead_optimization(
        input_compound, lead_optimizer, flavor_condition,
        top_k=top_k, max_decompositions=max_decompositions,
        pocket_condition=pocket_condition, gallery=gallery,
    )


## Run (Pocket-less Flavor Lead Optimization)


In [6]:
VANILLIN = "COc1cc(C=O)ccc1O"

report = lead_optimization_pocketless(
    VANILLIN, opt, ["sweet", "woody"], top_k=8,
    max_decompositions=4,
    gallery="vanillin_sweet_woody.pdf",
)
print(report)
print(f"gallery: {report.gallery}")
if len(report.failures):
    print(f"{len(report.failures)} suggestion(s) could not be assembled")

# the starting compound, on the same scale as everything below
print("\nINPUT", report.input_smiles)
for k, v in report.reference.items():
    print(f"   {k:<8} {v:.3f}" if v is not None else f"   {k:<8} n/a")

# d<prop> is the change against that input -- an absolute QED means little,
# +0.10 on the starting compound means something.
report.table[["product", "rgroup", "retrieval_score",
              "MW", "dMW", "logP", "dlogP", "QED", "dQED",
              "SAScore", "dSAScore", "NPScore", "dNPScore",
              "is_novel", "n_slots", "is_input"]]

<LeadOptimizationReport 'COc1cc(C=O)ccc1O' flavor=['sweet', 'woody'] 15 compounds>
gallery: vanillin_sweet_woody.pdf

INPUT COc1cc(C=O)ccc1O
   MW       152.149
   logP     1.213
   QED      0.648
   SAScore  1.822
   NPScore  0.854


,product,rgroup,retrieval_score,MW,dMW,logP,dlogP,QED,dQED,SAScore,dSAScore,NPScore,dNPScore,is_novel,n_slots,is_input
0,O=Cc1ccc(O)c(C=O)c1,*[CH]=O,48.820961,150.133,-2.016,1.0172,-0.1961,0.640028,-0.007716,2.356328,0.534277,0.781507,-0.072594,False,1,False
1,CCOc1cc(C=O)ccc1O,*OCC,46.778507,166.176,14.027,1.6034,0.3901,0.693574,0.045831,1.887087,0.065036,0.336861,-0.517241,False,1,False
2,COc1cc(CCO)ccc1O,*[CH2]CO,45.677845,168.192,16.043,0.9356,-0.2777,0.705641,0.057898,1.752230,-0.069821,0.981799,0.127697,False,1,False
3,COc1cc(C=O)ccc1O,*[CH]=O,45.449585,152.149,0.000,1.2133,0.0000,0.647744,0.000000,1.822051,0.000000,0.854102,0.000000,False,2,True
4,CCCc1ccc(O)c(OC)c1,*[CH2]CC,45.082699,166.220,14.071,2.3533,1.1400,0.746461,0.098718,1.600455,-0.221596,0.585980,-0.268121,False,1,False
5,COCc1cc(C=O)ccc1O,*[CH2]OC,44.989773,166.176,14.027,1.3511,0.1378,0.689251,0.041507,2.162709,0.340658,0.700537,-0.153565,False,1,False
6,CC(C)CCc1cc(C=O)ccc1O,*[CH2]CC(C)C,44.798645,192.258,40.109,2.7933,1.5800,0.744652,0.096908,2.155238,0.333187,0.901373,0.047272,False,1,False
7,C=Cc1ccc(O)c(OC)c1,*[CH]=C,44.575493,150.177,-1.972,2.0438,0.8305,0.698637,0.050894,1.930748,0.108697,1.011362,0.157260,False,1,False
8,CC(C)CC(=O)c1cc(C=O)ccc1O,*C(=O)CC(C)C,44.307800,206.241,54.092,2.4335,1.2202,0.607677,-0.040067,2.229621,0.407570,0.747077,-0.107025,False,1,False
9,COCc1ccc(O)c(OC)c1,*[CH2]OC,44.205376,168.192,16.043,1.5472,0.3339,0.743704,0.095960,1.679705,-0.142346,0.393655,-0.460447,False,1,False


## Run (Pocket-aware Flavor Lead Optimization)

## Pocket-conditioned variant

The default checkpoint has an UNTRAINED pocket path, so a pocket contributes
exactly zero there. `s3-pocket-supervised-s911012_best.pt` is supervised on the
243 tastepocket complexes (encoder and assembly head frozen, PCA-reduced pocket
+ projectors trained), so its pocket path is live.

Read the retrieval caveat before drawing conclusions: pocket conditioning
measured null on held-out receptors, and the cause is a granularity mismatch --
the pocket resolves chemotype while R-group retrieval needs exact fragments
(`docs/step3_pocket_capacity_2026-09-09.md`). The pocket moves the ranking here;
that is not the same as moving it correctly on a receptor the model never saw.


In [ ]:
from lead_optimization import pocket_embedding_from_structure

POCKET_CKPT = Path.home() / "checkpoints" / "molplatte" / "s3-pocket-supervised-s911012_best.pt"
STRUCTURE   = Path.home() / "datasets" / "tastepocket" / "structures" / "cif" / "8F76.cif"

# No MODEL_KWARGS: the architecture comes from the checkpoint sidecar, and the
# frozen PCA basis rides in the checkpoint itself -- the .npz that produced it
# is a training artefact, not needed at inference.
opt_pocket = LeadOptimizer.load(POCKET_CKPT, VOCAB, device="cuda")
opt_pocket.build_library(batch_size=1024)

pocket = pocket_embedding_from_structure(STRUCTURE, device="cuda")
print(f"pocket embedding {pocket.shape} from {STRUCTURE.name}")

report_pocket = lead_optimization_pocket(
    VANILLIN, opt_pocket, ["sweet"], pocket_condition=pocket, top_k=6,
    gallery="vanillin_pocket_8F76.pdf",
)
print(report_pocket)
print("pocket changed the ranking:", report_pocket.pocket_changed_ranking)

report_pocket.table[["product", "rgroup", "retrieval_score",
                     "MW", "dMW", "QED", "dQED", "SAScore", "dSAScore",
                     "is_novel", "n_slots"]]